In [1]:
# 1. get query from user
# 2. using Cosine Similarity find the most relevant chunk(2 or more)
# 3. make a prompt and feed to LLM with relevant chunk and query.

import ollama
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import json



In [2]:
with open("embedded_transcript.json", "r", encoding="utf-8") as json_file:
  data = json.load(json_file)["chunks"]

df = pd.DataFrame(data)

In [3]:

def fetch_relevant_chunk(user_query: str, dataframe, top_result=3):
  def embed_query(user_query):
    response = ollama.embed(
      model="nomic-embed-text",
      input=[user_query])
    
    return response.embeddings[0]
  
  embedded_query = embed_query(user_query)

  similarity = cosine_similarity(np.vstack(dataframe["embedding"]), [embedded_query]).flatten() # type: ignore

  top_similar_indx = similarity.argsort()[-top_result:][::-1]

  relevant_chunk = dataframe.loc[top_similar_indx][["text", "start", "duration"]]

  return relevant_chunk.to_json(orient="records")

In [4]:
rlvnt_chunk = fetch_relevant_chunk("what is hash table", df)

In [5]:
rlvnt_chunk

'[{"text":"These data structures, called hash tables, are essential to modern computing. Hash tables are really a general purpose tool that a program can then use on the fly. Hash tables are designed to insert or store elements of information in empty slots, and then search for these elements via queries. The big challenge in hash table design is the trade off","start":26.443,"duration":18.268},{"text":"The hash table can in principle do these very clever things. In order to make these insertions and these queries be very time efficient. The first hash table was invented in the 1950s by IBM engineer Hans-Peter Luhn. He had this insight that what you want to do when you\'re building a hash table, is you want to use randomness to your advantage. The hash function uses randomness to help select locations in memory.","start":92.258,"duration":25.317},{"text":"between the fullness of the hash table versus the insertion query time. There\'s this tension between time and space. In 2025, a stu

In [ ]:
from google import genai
from api_key import API_KEY

def LLM_response(query, metadata):
  prompt = f'''
  note: Given Below is a query and some data(contains- start time in seconds, duration in seconds, and text)
  and you have to answer the query using the data and mention the time stamp after converting seconds into minute(example if start time in seconds is 60.000 it will be  1:00 in minutes)
  important message: don't ask any question at end

  query: {query}

  data: {metadata}'''
  
  client = genai.Client(api_key=API_KEY)
  response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt)
  
  return response.text

In [9]:
LLM_response("what is hash table", rlvnt_chunk)

'A hash table is a general-purpose data structure essential to modern computing, designed to store elements of information in empty slots and allow for searching through queries (0:26). \n\nKey details about hash tables include:\n*   **Functionality:** They use randomness via a hash function to select locations in memory, ensuring that insertions and queries are very time-efficient (1:32).\n*   **Origin:** The first hash table was invented in the 1950s by IBM engineer Hans-Peter Luhn (1:32).\n*   **Design Challenges:** There is an inherent trade-off or "tension" in hash table design between space and time—specifically between how full the table is versus how quickly it can perform insertions and queries (0:48).'